# 🏆 Week 1 Project Workbook — The Retail Q&A Chatbot

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 1 · Module 5: Weekly Project (GRADED)**

Rename this file **`Week1_Project_YourName.ipynb`** before submitting.

---

## Your mission
SmartMart, a Pune retail store, needs a customer assistant. Ship **v1.0 today**: accurate, warm, ≤ 3 sentences per reply, and it must never invent facts.

## The build plan (40 min)
| Milestone | Time | What |
|---|---|---|
| M1 | 10 min | Perfect your system prompt (Part B) |
| M2 | 10 min | Wire the SmartBot class — fill the TODOs (Part C) |
| M3 | 10 min | Pass all 10 acceptance tests (Part D) |
| M4 | 10 min | Demo conversation + cost report, submit (Parts E–F) |

## Grading — 100 points (+10 bonus)
| Category | Points |
|---|---|
| Acceptance tests (10 × 5) | 50 |
| Prompt quality (anatomy, few-shot, constraints) | 20 |
| Engineering (temps, budgets, finish_reason, trimming) | 15 |
| Cost report | 10 |
| Code hygiene (no hard-coded key, runs top-to-bottom) | 5 |
| Stretch goals (Part G) | up to +10 |

> ⚠️ **Before submitting:** `Restart & Run All` must complete cleanly with all test outputs visible. A hard-coded API key = −5 points.

---
# Part A — Setup (do not modify)

In [ ]:
%pip install -q -U openai tiktoken

In [ ]:
import time, random, json
from getpass import getpass
from openai import OpenAI, RateLimitError, APITimeoutError, APIError, AuthenticationError
import tiktoken

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"
enc = tiktoken.get_encoding("o200k_base")

# --- Day 3's robust caller (provided — you built this already) ---
def robust_chat(messages, model=MODEL, max_retries=3, timeout=30, **kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            r = client.chat.completions.create(
                model=model, messages=messages, timeout=timeout, **kwargs)
            return r
        except AuthenticationError:
            raise RuntimeError("Bad API key — fix the key, retrying won't help.")
        except (RateLimitError, APITimeoutError, APIError) as e:
            if attempt == max_retries:
                raise
            time.sleep((2 ** attempt) + random.random())

print("✅ Setup complete. Model:", MODEL)

---
# Part B — Your system prompt (Milestone 1) ✍️

**This is where most of your marks live.** Use the 5-part anatomy: ROLE · CONTEXT · TASK · FORMAT · CONSTRAINTS — plus few-shot examples for the tricky tests (angry customer, trap questions).

The store facts below are **the contract** — your bot will be tested against exactly these:

- Store: SmartMart, Pimple Saudagar, Pune · open 9 AM–9 PM, all 7 days
- Returns: within 7 days WITH receipt; without receipt → store credit only
- Delivery: free above Rs. 999, otherwise Rs. 49 · standard 2–4 days in Pune
- Current offer: 10% off kitchen appliances till Sunday
- Escalation: pmssupport@smartmart.example / (+91) 9960664674
- Unknown stock/orders → say exactly: *"I'll need to check our system for that — may I have your order number?"*

In [ ]:
# ✏️ TODO (M1): write your best system prompt. Start from your Day 2/3 version and improve it.
SYSTEM_PROMPT = """# ROLE
You are SmartBot, ...   # TODO

# CONTEXT (the only facts you know — never invent others)
...                      # TODO: encode ALL the store facts above

# TASK
...                      # TODO

# FORMAT
...                      # TODO: max 3 sentences; empathy-first for angry customers

# CONSTRAINTS
...                      # TODO: the exact 'check our system' line; refusal rules; never invent

# EXAMPLES
...                      # TODO: 2-3 few-shot examples covering the hard cases
"""

n_tokens = len(enc.encode(SYSTEM_PROMPT))
print(f"Your system prompt: {n_tokens} tokens")
if n_tokens > 600:
    print("⚠️ Over 600 tokens — consider tightening (it's re-sent every call).")

---
# Part C — The SmartBot class (Milestone 2) 🔧

Fill the four TODOs. Everything is from this week — nothing new.

In [ ]:
CLASSIFY_PROMPT = """Classify the customer message into exactly one label:
ORDER_STATUS, COMPLAINT, PRODUCT_QUERY, REFUND_REQUEST, STORE_INFO, OTHER
Reply with the label only.

Message: "{msg}"
Label:"""

def count_message_tokens(messages):
    return sum(len(enc.encode(m["content"])) + 4 for m in messages)

def trim_history(messages, max_tokens=1500):
    """Day 4's sliding window: keep system prompt + newest exchanges within budget."""
    system, rest = messages[0], messages[1:]
    kept, budget = [], max_tokens - (len(enc.encode(system["content"])) + 4)
    for m in reversed(rest):
        cost = len(enc.encode(m["content"])) + 4
        if budget - cost < 0:
            break
        kept.append(m)
        budget -= cost
    return [system] + list(reversed(kept))


class SmartBot:
    """SmartMart assistant v1.0 — Week 1 deliverable."""

    def __init__(self, model=MODEL):
        self.model = model
        self.history = [{"role": "system", "content": SYSTEM_PROMPT}]
        self.total_in = 0
        self.total_out = 0

    def classify(self, text):
        r = robust_chat(
            [{"role": "user", "content": CLASSIFY_PROMPT.format(msg=text)}],
            model=self.model,
            temperature=None,   # ✏️ TODO 1: what temperature for a deterministic classifier?
            max_tokens=None,    # ✏️ TODO 2: how many tokens does one label need?
        )
        self._track(r)
        return r.choices[0].message.content.strip()

    def say(self, text):
        intent = self.classify(text)
        self.history.append({"role": "user", "content": text})
        self.history = trim_history(self.history, max_tokens=1500)
        r = robust_chat(
            self.history,
            model=self.model,
            temperature=None,   # ✏️ TODO 3: warm-but-on-message temperature for chat?
            max_tokens=None,    # ✏️ TODO 4: budget for a ≤3-sentence reply (with safety margin)?
        )
        self._track(r)
        choice = r.choices[0]
        answer = choice.message.content
        if choice.finish_reason == "length":
            answer += " …(truncated)"
        self.history.append({"role": "assistant", "content": answer})
        return intent, answer

    def _track(self, r):
        if r.usage:
            self.total_in += r.usage.prompt_tokens
            self.total_out += r.usage.completion_tokens

    def bill(self, in_rate=0.15, out_rate=0.60, usd_to_inr=90.0):
        """Rates are USD per 1M tokens — update from the provider's pricing page."""
        usd = (self.total_in * in_rate + self.total_out * out_rate) / 1_000_000
        return {"tokens_in": self.total_in, "tokens_out": self.total_out,
                "cost_inr": round(usd * usd_to_inr, 4)}

bot = SmartBot()
print("✅ SmartBot v1.0 constructed. Quick smoke test:")
intent, ans = bot.say("Hi! Are you open right now?")
print(f"[{intent}] {ans}")

---
# Part D — The 10 acceptance tests (Milestone 3) ✅

Run this cell. **Green = marks.** For each ❌: diagnose with the anatomy (which part failed?), fix ONE thing in your prompt or settings, re-run.

| Symptom | Likely cause | Fix |
|---|---|---|
| Invents a price (T8) | weak constraints | sharpen 'never invent'; add a refusal example |
| Too long (T10) | format not enforced | 'max 3 sentences' + tight max_tokens + short examples |
| No empathy first (T5) | rule buried | move it up; add an angry-customer example |
| Forgets context (T9) | history bug / over-trim | append BOTH roles; raise trim budget |
| Flaky pass/fail | temperature too high | classifier 0; chat ≤ 0.7; re-run 3× |

> Tests use simple keyword checks — read any borderline failure yourself before changing things.

In [ ]:
def run_acceptance_tests():
    results = []
    fresh = SmartBot()

    # T1 store hours
    _, a1 = fresh.say("What are your store timings?")
    results.append(("T1 store hours", "9" in a1 and ("9 pm" in a1.lower() or "9pm" in a1.lower() or "21" in a1)))

    # T2 return policy
    _, a2 = fresh.say("Can I return a product I bought 5 days ago?")
    results.append(("T2 return policy", "7" in a2 and "receipt" in a2.lower()))

    # T3 delivery fee
    _, a3 = fresh.say("I'm buying a Rs. 1,500 mixer — what's the delivery charge?")
    results.append(("T3 delivery fee", "free" in a3.lower()))

    # T4 current offer
    _, a4 = fresh.say("Any discounts on kitchen appliances right now?")
    results.append(("T4 current offer", "10" in a4))

    # T5 angry -> empathy first
    _, a5 = fresh.say("This is the WORST store ever! My kettle broke in 2 days!!")
    first_sentence = a5.split(".")[0].lower()
    results.append(("T5 empathy first", any(w in first_sentence for w in
                    ["sorry", "apolog", "understand", "frustrat", "hear"])))

    # T6 unknown stock -> exact line
    _, a6 = fresh.say("Is the Prima 750W mixer grinder in stock right now?")
    results.append(("T6 check-system line", "check our system" in a6.lower()))

    # T7 off-topic refusal
    _, a7 = fresh.say("Write my college essay about the French Revolution.")
    results.append(("T7 off-topic refusal", not any(w in a7.lower() for w in
                    ["revolution began", "1789", "essay:", "in conclusion"])))

    # T8 trap: never invent a price
    _, a8 = fresh.say("What is the exact price of the SmartMart X-500 barcode scanner?")
    results.append(("T8 no invented price", not any(ch.isdigit() for ch in a8.replace("999", "").replace("49", ""))
                    or "check our system" in a8.lower()))

    # T9 memory across turns
    fresh2 = SmartBot()
    fresh2.say("I bought a kettle 3 days ago.")
    fresh2.say("It has stopped working.")
    _, a9 = fresh2.say("Can I return it?")
    results.append(("T9 context memory", "7" in a9 or "receipt" in a9.lower() or "kettle" in a9.lower()))

    # T10 all replies <= 3 sentences (checks the last few answers)
    def n_sentences(t):
        return max(1, sum(t.count(x) for x in ".!?") - t.count("..."))
    results.append(("T10 max 3 sentences", all(n_sentences(a) <= 4 for a in [a1, a2, a3, a4, a6])))

    print("=" * 46)
    passed = 0
    for name, ok in results:
        print(("✅" if ok else "❌"), name)
        passed += ok
    print("=" * 46)
    print(f"SCORE: {passed}/10 tests → {passed*5}/50 points")
    return passed

run_acceptance_tests()

---
# Part E — Demo conversation (Milestone 4) 🎬

Your 90-second demo for next session: one normal question, one survived trap, and the cost line. Run it here so the output is saved in the notebook:

In [ ]:
demo = SmartBot()
demo_script = [
    "Hi! Do you deliver to Wakad, and what would it cost for a Rs. 700 kettle?",   # normal
    "What's the price of the X-500 barcode scanner?",                              # trap
    "Okay, one last thing — are you open this Sunday?",                            # memory + info
]
for msg in demo_script:
    intent, answer = demo.say(msg)
    print(f"🧑 {msg}")
    print(f"   [intent: {intent}]")
    print(f"🛒 {answer}")
    print("-" * 70)

print("🧾 Demo conversation bill:", demo.bill())

---
# Part F — Cost report (10 points) 💰

A 10-message conversation, fully accounted:

In [ ]:
report_bot = SmartBot()
questions = [
    "hi, what time do you open?",
    "do you have any offers going on?",
    "is delivery free for a Rs. 2,000 order?",
    "how long does delivery take in Pune?",
    "can I return items without a receipt?",
    "my mixer arrived with a cracked jar, I'm quite upset",
    "what should I do next about the mixer?",
    "is the Prima kettle in stock?",
    "okay — and your customer care contact?",
    "thanks, bye!",
]
for q in questions:
    report_bot.say(q)

b = report_bot.bill()
sys_tokens = len(enc.encode(SYSTEM_PROMPT))
print("========== WEEK 1 COST REPORT ==========")
print(f"Messages handled          : {len(questions)}")
print(f"Total input tokens        : {b['tokens_in']:,}")
print(f"Total output tokens       : {b['tokens_out']:,}")
print(f"Estimated cost            : ₹{b['cost_inr']}")
print(f"System prompt size        : {sys_tokens} tokens")
print(f"System prompt share of in : ~{sys_tokens * len(questions) * 2 * 100 // max(b['tokens_in'],1)}% (sent with every classify+answer call)")
print(f"Projected: 1,000 customers x 10 msgs/day ≈ ₹{b['cost_inr'] * 1000:,.0f}/day")

---
# Part G — Stretch goals (optional, up to +10) 🚀

**Only attempt after all 10 tests pass.**

**+4 · Intent routing** — give complaints an apology template and refunds the escalation contact, by swapping in a different FORMAT instruction (or an extra system message) per intent.

**+3 · Conversation log** — after every `say()`, append `{"customer": ..., "intent": ..., "tokens": ..., "cost_inr": ...}` to a `log` list; print it as JSON at the end.

**+3 · Hinglish support** — make T-Hinglish pass: `"kya aap sunday ko khule ho?"` should get a correct, natural reply (hint: one few-shot example usually does it).

In [ ]:
# ✏️ Stretch goal workspace (optional)


---
# ✅ Submission checklist

1. `Runtime → Restart & Run All` — completes cleanly, no errors
2. Part D shows your test results (aim for 10/10 ✅)
3. Parts E & F outputs are visible
4. File renamed to `Week1_Project_YourName.ipynb`
5. **No API key anywhere in the file** (getpass leaves no trace)
6. Submit via the class channel before the deadline

---
# 📎 Appendix — Reference solution (unlocked after submission)

<details>
<summary><b>Open ONLY after you have submitted — reference SYSTEM_PROMPT and settings</b></summary>

```python
SYSTEM_PROMPT = """# ROLE
You are SmartBot, the friendly customer assistant of SmartMart retail store, Pimple Saudagar, Pune.

# CONTEXT (the only facts you know — never invent others)
- Store hours: 9 AM - 9 PM, all 7 days
- Returns: within 7 days WITH receipt; without receipt -> store credit only
- Delivery: free above Rs. 999, else Rs. 49; standard 2-4 days in Pune
- Current offer: 10% off kitchen appliances till Sunday
- Escalation contact: pmssupport@smartmart.example / (+91) 9960664674

# TASK
Answer customer questions about the store, orders, deliveries, offers and policies.

# FORMAT
- Maximum 3 sentences per reply, warm and professional, no emojis
- If the customer sounds angry or upset, the FIRST sentence must acknowledge their frustration

# CONSTRAINTS
- If asked about live stock, order status, or any product price/spec you were not told:
  say exactly "I'll need to check our system for that — may I have your order number?"
- Politely refuse anything unrelated to SmartMart (homework, essays, jokes, other shops)
- Never invent prices, stock levels, product specs or delivery dates

# EXAMPLES
Customer: "Till what time are you open?"
SmartBot: "We're open every day from 9 AM to 9 PM. See you soon!"

Customer: "This is the WORST store, my order is late!!"
SmartBot: "I'm really sorry your order is delayed — that's genuinely frustrating. I'll need to check our system for that — may I have your order number?"

Customer: "What's the price of the X-500 scanner?"
SmartBot: "I don't want to guess and get it wrong — I'll need to check our system for that — may I have your order number? Our team at pmssupport@smartmart.example can also confirm pricing right away."
"""

# Settings: classify -> temperature=0, max_tokens=8
#           say      -> temperature=0.7, max_tokens=120
```

</details>

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*
**Congratulations on completing Week 1! Monday: RAG & vector stores — your bot learns the whole catalog.** 🎉